# 5. Model training and feature experiments

**Current mode: exploratory local training using your saved four-month features.** Run Steps 1–4 sequentially in a fresh `.venv` kernel. No satellite downloads or Earth Engine sign-in are needed for this mode.

It trains three scikit-learn Random Forests with **1,000 trees, seed 42**, comparing interval-only, interval-plus-shape, and shape-only predictors. Source parcel groups stay entirely within one partition: approximately 60% training, 20% validation, and 20% test groups per crop. Pixel proportions may differ because parcels have different sizes. Configuration and split are saved before fitting.

Feature selection uses validation macro-F1. The same held-out test parcels evaluate all designs after that selection. Accuracy measures agreement with the external source labels; it does not establish local Philippine accuracy, verify field identities, or validate estimated crop cycles. Cabbage comes from a different source/year, so dataset differences can influence scores.

Set `training_mode = "production"` later to use the strict Earth Engine feature handoff. That path still requires bilinear imagery, frozen settings, and independently reviewed local fields.

**Active run: `bilinear_expanded`.** Local exploratory training reads only this run’s complete pixel tables, computes the three feature designs, and saves models and accuracy under `seasons/existing_parcel_exploratory_bilinear/`. A new parcel split is locked for this collection; pixels from one source parcel stay together. Old datasets, splits, and models remain in their original folders. This notebook does not download imagery.


## 1. Choose training mode and saved inputs

Leave `training_mode = "existing_parcel_exploratory"` for the current files. `input_feature_paths = None` selects complete pixel tables only from `bilinear_expanded`, with no fallback to an older run. No imagery is opened or downloaded.

Output: `Training mode` identifies the local exploratory or production path. Production input/output paths are printed here; the exploratory destination is resolved in Step 2.

Production settings remain available: use `feature_handoff.json`, or explicitly enable the older band-comparison compatibility path. Exploratory mode never relabels existing nearest-neighbor imagery as bilinear.

The new collection writes to `existing_parcel_exploratory_bilinear/`. `previous_split_path = None` creates a deterministic parcel split for this collection; subsequent reruns must match its locked inputs and split. Historical scores use different samples and are not directly comparable.


In [ ]:
from pathlib import Path
import json
import os
from crop_pipeline.windows import seasons as season_workflow
from crop_pipeline.features import experiments as feature_experiments
from crop_pipeline.training import earth_engine as earth_engine_training
from crop_pipeline.training import exploratory as exploratory_training

# Notebooks live in notebooks/; run from the repository root so relative paths resolve.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
settings_path = Path("settings.local.json")
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
report_folder = (
    Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).expanduser().resolve()
)
# Use "production" later for the strict bilinear / independent-local-field experiments.
training_mode = "existing_parcel_exploratory"
input_feature_paths = None  # Automatically choose the saved complete pixel tables.
exploratory_run_name = "existing_parcel_exploratory_bilinear"
analysis_run_name = "bilinear_expanded"  # Keep the previous pilot results separate.
season_folder = report_folder.parent / "analysis" / analysis_run_name / "seasons"
handoff_path = season_folder / "feature_input" / "feature_handoff.json"
# Preserve the earlier run's train/validation/test parcel assignments.
previous_split_path = None  # Lock a new parcel split for this independently sampled collection.
training_output_folder = handoff_path.parent.parent
allow_legacy_band_comparison = False
authenticate_now = False
print("Training handoff:", handoff_path)
print("Output folder:", training_output_folder)

if training_mode not in {"existing_parcel_exploratory", "production"}:
    raise ValueError("Unknown training mode.")
print("Training mode:", training_mode)


## 2. Verify inputs and lock the parcel split

Exploratory mode rejects missing groups, conflicting crop labels, shared locations/known field aliases across groups, duplicate sample identities, and previously reserved partitions. It requires at least five groups per crop and all three crops in each partition. It never randomly splits a parcel's pixels.

Output: `Using saved feature tables` identifies the actual inputs. `Split counts` reports pixels and source parcel groups by crop and partition. `Prepared pixel windows` is the total eligible sample count. The split and input hashes are locked under `existing_parcel_exploratory/`; changed inputs require a new output folder rather than silently changing the holdout.

Production mode verifies its feature handoff hashes and keeps its existing freeze and local-field requirements.


In [ ]:
if training_mode == "existing_parcel_exploratory":
    legacy_input = False
    if input_feature_paths is None:
        input_feature_paths = exploratory_training.existing_feature_tables(report_folder, analysis_run_name=analysis_run_name)
    training_output_folder = season_folder / exploratory_run_name
    partitioned_rows, configuration, split_counts, handoff_path = exploratory_training.prepare_existing_inputs(
        input_feature_paths, training_output_folder, previous_split_path=previous_split_path
    )
    preprocessing_coverage = {"split_summary": split_counts, "evaluation_scope": "held_out_source_parcel_labels",
                              "imagery_policy": configuration["imagery_policy"]}
    handoff_manifest = {"status": "prepared"}
    print("Using saved feature tables:", [str(path) for path in input_feature_paths])
    print("Split counts:", split_counts)
else:
    legacy_input = handoff_path.name == "training_handoff.json"
    if legacy_input and not allow_legacy_band_comparison:
        raise ValueError(
            "Run 04_feature_engineering.ipynb and choose feature_handoff.json for the revised experiments."
        )
    loader = (
        season_workflow.load_training_handoff
        if legacy_input
        else feature_experiments.load_feature_handoff
    )
    partitioned_rows, configuration, preprocessing_coverage, handoff_manifest = loader(handoff_path)
    window_policy = configuration.get("window_policy", "reference_supported_only")
    training_output_folder = handoff_path.parent.parent
    if window_policy == "exploratory_external_estimates":
        training_output_folder = training_output_folder / "exploratory_estimated"
print("Preparation status:", handoff_manifest["status"])
print("Prepared pixel windows:", len(partitioned_rows))
print("Configured crops:", configuration["study_classes"])
print("Window policy:", configuration.get("window_policy", "reference_supported_only"))
print("Model output folder:", training_output_folder)


## 3. Fit models, select on validation, and check held-out accuracy

Exploratory forests use 1,000 trees, seed 42, square-root predictors per split, minimum leaf population 2, bootstrap sampling with `max_samples=0.5`, and source-parcel confidence sample weights. All reported labels receive the same confidence; inverse parcel pixel counts reduce domination by large parcels. This is a local scikit-learn implementation, not an identical implementation of Earth Engine bagging.

All three designs are trained and validation scores saved before any test scoring. The saved selection uses validation macro-F1 only. Test predictions use the same held-out samples for every design.

Output: each feature design's held-out **accuracy**, **macro-F1**, and **balanced accuracy**. These are pixel-level metrics; equally weighted mean parcel accuracy is also saved. Per-crop precision, recall, F1, support and confusion matrices are in `experiment_metrics.json`. A high score does not establish transfer to Philippine fields.

Production mode retains its nine Earth Engine comparisons and shared independent local test set. No source downloads occur in either mode.


In [ ]:
model_folder = training_output_folder / "feature_models"
model_artifacts = []
comparison_metrics = {}
feature_selections = {}
if training_mode == "existing_parcel_exploratory":
    model_folder = training_output_folder
    comparison_metrics, feature_selections = exploratory_training.run_experiments(
        partitioned_rows, configuration, training_output_folder
    )
    model_artifacts = [path for path in training_output_folder.rglob("*") if path.is_file() and "training_package" not in path.parts]
    print("Feature selections:", feature_selections)
    for design, metrics in comparison_metrics["test"].items():
        print(f"{design}: held-out accuracy={metrics['accuracy']:.3f}, macro-F1={metrics['macro_f1']:.3f}, balanced accuracy={metrics['balanced_accuracy']:.3f}")
elif not partitioned_rows or handoff_manifest["status"] != "prepared":
    print(
        "Training pending: supply complete inputs, independent local fields and frozen development settings."
    )
elif legacy_input:
    model_folder = training_output_folder / "models"
    for feature_count, include_indices in [(40, False), (64, True)]:
        destination = model_folder / f"features_{feature_count}"
        comparison_metrics[str(feature_count)] = season_workflow.train_comparisons(
            partitioned_rows, configuration, destination, include_indices=include_indices
        )
    model_artifacts = [path for path in model_folder.rglob("*") if path.is_file()]
else:
    import ee

    if authenticate_now:
        ee.Authenticate()
    ee.Initialize(project=local_settings.get("earth_engine_project", "YOUR_EARTH_ENGINE_PROJECT"))
    comparison_metrics, feature_selections = earth_engine_training.train_feature_experiments(
        partitioned_rows, configuration, model_folder, ee
    )
    model_artifacts = [path for path in model_folder.rglob("*") if path.is_file()]
    print("Feature selections:", feature_selections)
season_workflow.save_json(
    training_output_folder / "training_status.json",
    {
        "status": "completed" if comparison_metrics else "pending_inputs",
        "legacy_band_comparison": legacy_input,
        "training_mode": training_mode,
        "feature_selections": feature_selections,
    },
)


## 4. Package the result

Output: `Training status: completed` means the comparisons ran; it does not mean any target accuracy was achieved. `Training package` identifies the provenance inventory with input hashes, configuration and output files.

Exploratory models are local `model.joblib` artifacts containing the fitted forest, ordered predictors, and configuration. Only load trusted joblib files. Parcel split, validation/test predictions, metrics, and validation-based feature selection are saved separately from production outputs. Earth Engine production models continue to use fitted-tree JSON.


In [ ]:
training_coverage = {
    **preprocessing_coverage,
    "sample_counts": season_workflow.sample_counts(partitioned_rows),
    "training_status": "completed" if comparison_metrics else "pending_inputs",
    "training_handoff": str(handoff_path.resolve()),
    "feature_selections": feature_selections,
}
if training_mode == "existing_parcel_exploratory":
    handoff_input_paths = [handoff_path] + list(input_feature_paths) + [training_output_folder / "parcel_split.csv"]
elif legacy_input:
    handoff_input_paths = [handoff_path] + [
        handoff_path.parent / record["filename"] for record in handoff_manifest["files"].values()
    ]
else:
    handoff_input_paths = [handoff_path] + [
        handoff_path.parent / filename for filename in handoff_manifest["files"]
    ]
training_manifest = season_workflow.package_run(
    training_output_folder / "training_package",
    configuration,
    input_paths=handoff_input_paths
    + [
        Path("src/crop_pipeline/features/experiments.py"),
        Path("src/crop_pipeline/training/earth_engine.py"),
        Path("src/crop_pipeline/training/exploratory.py"),
        Path("src/crop_pipeline/windows/seasons.py"),
        Path("requirements-notebooks.txt"),
        Path("notebooks/05_model_training.ipynb"),
    ],
    artifact_paths=model_artifacts + [training_output_folder / "training_status.json"],
    coverage=training_coverage,
)
print("Training status:", training_coverage["training_status"])
print("Training package:", training_output_folder / "training_package")


## Interpret the exploratory result

Inspect `experiment_metrics.json`, `feature_selection.json`, and `parcel_split.csv` under the printed model output folder (`analysis/bilinear_expanded/seasons/existing_parcel_exploratory_bilinear/`). Report the validation-selected design and its held-out result, sample counts and parcel counts. Do not choose a design or repeatedly tune parameters using test scores.

Source parcel grouping prevents the same recorded parcel from entering multiple partitions, but it does not prove geographic independence between nearby parcels or independently verify their identities. The holdout uses reported external labels and automatically estimated windows. It is a prototype evaluation, not the thesis's independent local evaluation. See [training requirements](../docs/training.md).
